# CoffeeFG-YOLO26 v2 — A0 nyata

Notebook bersih untuk screening validation-first. Urutan: setup → restore A0 train/val → preflight → D0/D1 → diagnostic. Test tidak diekstrak.

In [ ]:
# 1. SETUP BERSIH
import os, shutil, subprocess, sys
from collections import deque
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH,
    'https://github.com/ediprin/coffee-bean-detection.git', str(REPO),
], text=True, capture_output=True)
if clone.returncode != 0:
    print(clone.stdout)
    print(clone.stderr)
    raise RuntimeError(f'Git clone gagal: return code {clone.returncode}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
SRC = REPO / 'src'
sys.path.insert(0, str(SRC))
os.environ['PYTHONPATH'] = str(SRC) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.chdir(REPO)
import coffee_detector
import torch
print('IMPORT :', coffee_detector.__file__)
print('COMMIT :', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('GPU    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'TIDAK ADA')
assert torch.cuda.is_available(), 'Aktifkan GPU runtime sebelum melanjutkan.'

def run_live(command, log_path):
    command = [str(item) for item in command]
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('MENJALANKAN:', ' '.join(command), flush=True)
    tail = deque(maxlen=120)
    env = os.environ.copy()
    env['PYTHONPATH'] = str(SRC) + os.pathsep + env.get('PYTHONPATH', '')
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            command, cwd=REPO, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush(); tail.append(line)
    code = process.wait()
    if code != 0:
        print('\n=== 120 BARIS TERAKHIR ===\n' + ''.join(tail))
        raise RuntimeError(f'Proses gagal ({code}). Log: {log_path}')
    return code

In [ ]:
# 2. DRIVE DAN ARTEFAK
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')

preferred = [
    DRIVE / 'Coffee_Bean_Detection/bundles/sni21-vadcp-pilot-bundle/A0_real.tar',
    DRIVE / 'coffee-bean-detection/sni21-vadcp-pilot-bundle/A0_real.tar',
    DRIVE / '02_RISET_DAN_PROYEK/Coffee_Bean_Detection/bundles/sni21-vadcp-pilot-bundle/A0_real.tar',
]
A0_ARCHIVE = next((path for path in preferred if path.is_file()), None)
if A0_ARCHIVE is None:
    roots = [DRIVE, Path('/content/drive/.shortcut-targets-by-id')]
    matches = []
    for root in roots:
        if root.is_dir():
            matches.extend(path for path in root.rglob('A0_real.tar') if path.is_file())
    matches = sorted(set(matches))
    assert matches, 'A0_real.tar tidak ditemukan di My Drive/shortcut.'
    A0_ARCHIVE = matches[0]
PROJECT_DRIVE = DRIVE / 'Coffee_Bean_Detection'
OUTPUT_ROOT = PROJECT_DRIVE / 'experiments/coffee-fg-quick10-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('ARSIP  :', A0_ARCHIVE)
print('OUTPUT :', OUTPUT_ROOT)

In [ ]:
# 3. RESTORE TRAIN+VAL DAN PREFLIGHT KETAT
import yaml
from coffee_detector.archive_sni21_pilot import restore_real_a0_development
from ultralytics.data.utils import check_det_dataset

DATA_ROOT = restore_real_a0_development(
    A0_ARCHIVE, '/content/sni21-a0-development'
)
yaml_path = DATA_ROOT / 'data.yaml'
payload = yaml.safe_load(yaml_path.read_text(encoding='utf-8'))
assert Path(payload['path']).resolve() == DATA_ROOT.resolve(), payload
assert payload['train'] == 'train/images', payload
assert payload['val'] == 'val/images', payload
assert 'test' not in payload, 'Test harus tetap terkunci.'
train_count = sum(path.is_file() for path in (DATA_ROOT / 'train/images').rglob('*'))
val_count = sum(path.is_file() for path in (DATA_ROOT / 'val/images').rglob('*'))
assert train_count > 0 and val_count > 0
checked = check_det_dataset(str(yaml_path))
print('TRAIN  :', train_count, 'gambar')
print('VAL    :', val_count, 'gambar')
print('YAML   :', yaml_path)
print('TEST   : TIDAK DIEKSTRAK')
print('PREFLIGHT ULTRALYTICS: BERHASIL')

## Tahap 1: D0Q versus D1Q (quick-10)

D0Q adalah YOLO26n P3–P5; D1Q menambahkan P2. Keduanya memakai seed 42 dan 10 epoch. Ini screening penghemat kuota, bukan hasil final. Output berada di Drive dan runner melanjutkan `last.pt` jika runtime terputus.

In [ ]:
# 4. TRAIN/RESUME D0Q DAN D1Q (10 EPOCH)
AUDIT = OUTPUT_ROOT / 'val_reports/dataset_audit.json'
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_coffee_fg_screening',
    '--data-root', DATA_ROOT, '--output-root', OUTPUT_ROOT,
    '--models', 'D0Q', 'D1Q', '--seeds', '42',
    '--evaluation-split', 'val', '--device', '0',
]
if AUDIT.is_file():
    command += ['--verified-audit', AUDIT]
run_live(command, OUTPUT_ROOT / 'logs/stage1_d0q_d1q.log')
assert (OUTPUT_ROOT / 'D0Q_seed42/weights/best.pt').is_file()
assert (OUTPUT_ROOT / 'D1Q_seed42/weights/best.pt').is_file()
print('D0Q DAN D1Q SELESAI')

In [ ]:
# 5. DIAGNOSTIC VALIDATION
DIAGNOSTIC = OUTPUT_ROOT / 'val_reports/diagnostic_seed42.json'
command = [
    sys.executable, '-u', '-m', 'coffee_detector.analysis.coffee_fg_diagnostics',
    '--p3-checkpoint', OUTPUT_ROOT / 'D0Q_seed42/weights/best.pt',
    '--p2-checkpoint', OUTPUT_ROOT / 'D1Q_seed42/weights/best.pt',
    '--data-root', DATA_ROOT, '--output', DIAGNOSTIC, '--split', 'val',
    '--candidate-counts', '50', '100', '300', '500',
    '--max-det', '500', '--device', '0',
]
run_live(command, OUTPUT_ROOT / 'logs/diagnostic_seed42.log')
import json
decision = json.loads(DIAGNOSTIC.read_text(encoding='utf-8'))['decision']
print(json.dumps(decision, indent=2, ensure_ascii=False))
if decision['classification_refinement_rational']:
    print('KIRIM HASIL INI. Kandidat:', decision['recommended_refiners'])
else:
    print('STOP: CoffeeFG refiner tidak rasional pada A0 validation.')

## Tahap 2 dikunci

Jangan melatih R0/R1 atau R2/R3 sebelum mengirim keluaran diagnostic. Notebook ini sengaja berhenti di sini.